In [1]:
import json
import sys
from rich import print as rprint
from pathlib import Path
import re

nb_dir = Path.cwd()

project_root = nb_dir.parent.parent

sys.path.insert(0, str(project_root))

In [2]:
b2p_broken_file = Path(project_root / "data_reload/broken-b2p-loading.json")

db_people_file = Path(project_root / "data/from db/people_db.json")

with open(b2p_broken_file, "r") as f:
   broken = json.load(f)

with open(db_people_file, "r") as f:
   db_people = json.load(f)

uid_dict = {
   u["unified_id"]:{"person_id": u["person_id"],
            "unified_id": u["unified_id"]}
              for u in db_people}

rprint(dict(list(uid_dict.items())[:2]))

rprint(broken[:1])

{
    'gockel_gabriele': {'person_id': 1, 'unified_id': 'gockel_gabriele'},
    'allram_josef': {'person_id': 2, 'unified_id': 'allram_josef'}
}

[
    {
        'book_id': 1988,
        'composite_id': 'erstausgaben_1592_64_78',
        'person_id': 2513,
        'unified_id': 'bartens_daniela',
        'display_name': 'BARTENS, Daniela',
        'sort_order': 1,
        'is_author': False,
        'is_editor': True,
        'is_contributor': False,
        'is_translator': False
    }
]

In [5]:
b2p_fixed_list = []
not_found = []
nf_count = 0
uid_mismatch = []

for entry in broken:
# for entry in broken[:50]:
    unified_id = entry["unified_id"]

    if not unified_id in uid_dict:
        nf_count += 1
        not_found.append(entry)
        continue

    uid = uid_dict[unified_id]["unified_id"]
    person_id = uid_dict[unified_id]["person_id"]
    # rprint(uid, person_id)

    if unified_id == uid:
        b2p_fixed_list.append({
            **entry,
            "person_id": person_id,
            "person_id_old": entry["person_id"]
        })
    else:
        uid_mismatch.append(entry)

b2p_loading_file = Path(project_root / "data_reload/b2p_loading_file.json")

with open(b2p_loading_file, "w") as f:
    json.dump(b2p_fixed_list, f, ensure_ascii=False, indent=2)



rprint(f"""
=== LOG ===
b2p before: {len(broken)}
b2p fixed: {len(b2p_fixed_list)}
mismatch: {len(uid_mismatch)}

=== DONE ===
""")


=== LOG ===
b2p before: 11624
b2p fixed: 11624
mismatch: 0

=== DONE ===